# BAA10Y analysis-agent integration tests

In [1]:
from __future__ import annotations
import warnings
warnings.filterwarnings('ignore')

import json
import sys
from pathlib import Path
from typing import Any

import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "aieng-forecasting").is_dir() and (candidate / "implementations").is_dir():
            return candidate
    raise RuntimeError(
        "Repository root not found. Run this notebook from inside the "
        "agentic-forecasting repository."
    )


REPO_ROOT = find_repo_root(Path.cwd().resolve())
IMPLEMENTATIONS_ROOT = REPO_ROOT / "implementations"

if str(IMPLEMENTATIONS_ROOT) not in sys.path:
    sys.path.insert(0, str(IMPLEMENTATIONS_ROOT))

AS_OF = "2026-07-30"
HORIZONS = (1, 5, 21)
PRIMARY_HORIZON = 21
PRIMARY_QUESTION = (
    "Using information available as of July 30, 2026, analyze whether "
    "recent credit-market drivers support BAA10Y widening or tightening "
    "over the next 21 business days."
)

# Opt in only after the no-model tests pass.
RUN_AGENT_TESTS = True

print("Repository root:", REPO_ROOT)
print("As-of date:", AS_OF)
print("Horizons:", HORIZONS)

Repository root: /home/coder/agentic-forecasting
As-of date: 2026-07-30
Horizons: (1, 5, 21)


In [2]:
from aieng.forecasting.evaluation.task import ForecastingTask
from aieng.forecasting.methods.agentic import build_adk_agent
from aieng.forecasting.methods.agentic.adk_runner import (
    AdkTextRunner,
    AdkTextRunnerConfig,
)
from BAA10Y_forecasting.analyst_agent import agent as agent_module
from BAA10Y_forecasting.analyst_agent.agent import (
    BAA10Y_ANALYST_COVARIATE_SERIES_IDS,
    build_baa10y_multitask_news_config,
    load_baa10y_analysis_payload,
)
from BAA10Y_forecasting.data import (
    baa10y_change_series_id,
    build_baa10y_multivariate_service,
)
from BAA10Y_forecasting.tasks import (
    BAA10YMultitaskPromptBuilder,
    TASK_SPECS,
)


def tool_name(tool: Any) -> str:
    name = getattr(tool, "name", None)
    if name:
        return str(name)
    func = getattr(tool, "func", None)
    if func is not None:
        return str(getattr(func, "__name__", type(tool).__name__))
    return str(getattr(tool, "__name__", type(tool).__name__))



print("Loaded agent module:", agent_module.__file__)
print("Loader function:", load_baa10y_analysis_payload.__name__)

Loaded agent module: /home/coder/agentic-forecasting/implementations/BAA10Y_forecasting/analyst_agent/agent.py
Loader function: load_baa10y_analysis_payload


## Test 1 — configuration and final ADK tool registration

This is the fastest diagnostic for the earlier error: `Tool ... not found. Available tools: search_web`. Both assertions must pass before running any model call.

In [3]:
config = build_baa10y_multitask_news_config()
configured_tool_names = [tool_name(tool) for tool in config.function_tools]

adk_agent = build_adk_agent(config)
final_tool_names = [tool_name(tool) for tool in adk_agent.tools]

registration_table = pd.DataFrame(
    [
        {
            "layer": "AgentConfig.function_tools",
            "tools": configured_tool_names,
        },
        {
            "layer": "Built ADK agent.tools",
            "tools": final_tool_names,
        },
    ]
)
display(registration_table)

assert "load_baa10y_analysis_payload" in configured_tool_names, (
    "Loader is missing from AgentConfig.function_tools. Update "
    "build_baa10y_multitask_news_config() in analyst_agent/agent.py."
)
assert "load_baa10y_analysis_payload" in final_tool_names, (
    "Loader did not reach the final ADK agent. Confirm the active agent.py "
    "path above and fully restart ADK Web."
)
assert "search_web" in final_tool_names

print("PASS: search_web and load_baa10y_analysis_payload are registered.")

,layer,tools
0,AgentConfig.function_tools,[load_baa10y_analysis_payload]
1,Built ADK agent.tools,"[search_web, load_baa10y_analysis_payload]"


PASS: search_web and load_baa10y_analysis_payload are registered.


## Test 2 — verify parity with `tasks.py`

The interactive loader should reuse `BAA10YMultitaskPromptBuilder`. This test independently builds the same `ForecastContext` and compares the core payload fields.

In [4]:
cutoff = pd.Timestamp(AS_OF).normalize()
service_end = str((cutoff + pd.Timedelta(days=1)).date())
target_series_id = baa10y_change_series_id(PRIMARY_HORIZON)

service = build_baa10y_multivariate_service(
    windows=(PRIMARY_HORIZON,),
    covariate_series_ids=BAA10Y_ANALYST_COVARIATE_SERIES_IDS,
    strict_covariates=False,
    refresh=False,
    end=service_end,
)
context = service.context(cutoff.to_pydatetime())
task = ForecastingTask(
    task_id=f"baa10y_interactive_{PRIMARY_HORIZON}b",
    target_series_id=target_series_id,
    horizons=[PRIMARY_HORIZON],
    frequency="B",
    description=PRIMARY_QUESTION,
)
builder_payload = json.loads(
    BAA10YMultitaskPromptBuilder(task_spec=PRIMARY_QUESTION)(
        task=task,
        context=context,
    )
)
loader_payload = load_baa10y_analysis_payload(
    question=PRIMARY_QUESTION,
    as_of=AS_OF,
    horizon_business_days=PRIMARY_HORIZON,
)

if loader_payload.get("status") != "ok":
    raise RuntimeError(
        "Parity-test loader call failed: "
        + loader_payload.get(
            "message",
            str(loader_payload),
        )
    )
fields_to_compare = [
    "task",
    "task_spec",
    "as_of",
    "origin_target_change_bps",
    "target_history_csv",
    "covariate_history",
]
comparison_rows = []
for field in fields_to_compare:
    matches = loader_payload[field] == builder_payload[field]
    comparison_rows.append({"field": field, "matches": matches})
    assert matches, f"Loader and tasks.py builder differ for {field!r}"

display(pd.DataFrame(comparison_rows))
print("PASS: interactive and predictor payload construction are aligned.")

,field,matches
0,task,True
1,task_spec,True
2,as_of,True
3,origin_target_change_bps,True
4,target_history_csv,True
5,covariate_history,True


PASS: interactive and predictor payload construction are aligned.


## Test 3 — task-style prompt suite

These prompts exercise statistical analysis, driver interpretation defined in the SKILL.md

In [5]:
SKILL_TEST_CASES = [
    # ============================================================
    # statistical-analysis: three supported diagnostic patterns
    # ============================================================
    {
        "case": "statistical_volatility_regime",
        "skill": "statistical-analysis",
        "type": "volatility_regime",
        "prompt": (
            "Using information available as of July 30, 2026, "
            "classify the volatility regime of the 21-business-day "
            "BAA10Y change series as low, normal, elevated, or extreme. "
            "Report the current 30-observation volatility, median "
            "historical rolling volatility, and volatility ratio."
        ),
    },
    {
        "case": "statistical_anomaly_detection",
        "skill": "statistical-analysis",
        "type": "anomaly_detection",
        "prompt": (
            "Using information available as of July 30, 2026, "
            "determine whether the latest 21-business-day BAA10Y "
            "spread-change observation is anomalous. Report the latest "
            "value, rolling standard deviation, z-score, and whether "
            "the absolute z-score exceeds 2.5."
        ),
    },
    {
        "case": "statistical_window_selection",
        "skill": "statistical-analysis",
        "type": "analysis_window",
        "prompt": (
            "Using information available as of July 30, 2026, "
            "determine whether the statistical analysis should use "
            "15, 30, or 45 recent observations. First calculate the "
            "volatility regime and latest-observation z-score, then "
            "report the selected window, recent median, recent "
            "standard deviation, and selection reason."
        ),
    },

    # ============================================================
    # credit-driver-analysis: three supported analysis patterns
    # ============================================================
    {
        "case": "driver_movement_summary",
        "skill": "credit-driver-analysis",
        "type": "driver_movements",
        "prompt": (
            "Using information available as of July 30, 2026, first "
            "use statistical analysis to select the appropriate "
            "15-, 30-, or 45-observation window. Then summarize the "
            "recent movements of all available BAA10Y credit and "
            "market drivers. Report each driver's treatment, latest "
            "value, net movement, movement z-score, and whether it "
            "is scorable or context only."
        ),
    },
    {
        "case": "driver_evidence_translation",
        "skill": "credit-driver-analysis",
        "type": "widening_tightening_evidence",
        "prompt": (
            "Using information available as of July 30, 2026, first "
            "select the appropriate statistical analysis window. "
            "Then translate each available market driver into "
            "widening, tightening, neutral, or context-only evidence. "
            "Report the credit score and classification for each "
            "scorable driver. Do not double-count observed and proxy "
            "HYOAS or VIX return and VIX level."
        ),
    },
    {
        "case": "combined_driver_conclusion",
        "skill": "credit-driver-analysis",
        "type": "combined_conclusion",
        "prompt": (
            "Using information available as of July 30, 2026, first "
            "determine the volatility regime, anomaly status, and "
            "appropriate analysis window. Then combine the available "
            "credit-driver evidence. Report the combined score, "
            "overall widening, tightening, mixed, or neutral signal, "
            "confidence level, strongest widening evidence, strongest "
            "tightening evidence, and important context-only factors."
        ),
    },
]


In [7]:
# Create one runner. Each question gets a fresh ADK session.
skill_test_runner = AdkTextRunner(
    adk_agent,
    config=AdkTextRunnerConfig(
        app_name="baa10y_skill_tests",
        default_user_id="notebook_tester",
        fresh_session_per_message=True,
        enable_langfuse_tracing=False,
    ),
)


test_results = []
full_responses = {}


for test_case in SKILL_TEST_CASES:
    case_name = test_case["case"]

    try:
        response = await skill_test_runner.run_text_async(
            test_case["prompt"]
        )

        full_responses[case_name] = response

        test_results.append(
            {
                "case": case_name,
                "skill": test_case["skill"],
                "type": test_case["type"],
                "status": "completed",
                "response_length": len(response),
            }
        )

    except Exception as exc:
        error_message = (
            f"{type(exc).__name__}: {exc}"
        )

        full_responses[case_name] = error_message

        test_results.append(
            {
                "case": case_name,
                "skill": test_case["skill"],
                "type": test_case["type"],
                "status": "error",
                "response_length": 0,
                "error": error_message,
            }
        )

        print(f"ERROR: {case_name}: {error_message}")

    finally:
        # Close the runner only after every test case has run.
        await skill_test_runner.aclose()

results_df = pd.DataFrame(test_results)
display(results_df)


for test_case in SKILL_TEST_CASES:
    case_name = test_case["case"]
    response = full_responses.get(
        case_name,
        "No response returned.",
    )

    display(
        Markdown(
            f"## {case_name}\n\n"
            f"**Skill:** `{test_case['skill']}`  \n"
            f"**Question type:** "
            f"`{test_case['type']}`\n\n"
            f"### Response\n\n"
            f"{response}"
        )
    )

,case,skill,type,status,response_length
0,statistical_volatility_regime,statistical-analysis,volatility_regime,completed,1155
1,statistical_anomaly_detection,statistical-analysis,anomaly_detection,completed,1020
2,statistical_window_selection,statistical-analysis,analysis_window,completed,1851
3,driver_movement_summary,credit-driver-analysis,driver_movements,completed,3040
4,driver_evidence_translation,credit-driver-analysis,widening_tightening_evidence,completed,3226
5,combined_driver_conclusion,credit-driver-analysis,combined_conclusion,completed,2902


## statistical_volatility_regime

**Skill:** `statistical-analysis`  
**Question type:** `volatility_regime`

### Response

Based on the data available as of July 30, 2026, the 21-business-day BAA10Y change series is currently in a **normal** volatility regime.

### Volatility Analysis
The following metrics are derived from the 21-business-day change series, evaluated using the requested 30-observation rolling window:

*   **Current 30-observation volatility:** 6.42 basis points
*   **Median historical rolling volatility:** 6.64 basis points
*   **Volatility ratio:** 0.97

### Classification Methodology
The volatility regime is classified by comparing the current 30-observation rolling volatility to the historical median of the same series:
*   **Low:** Ratio < 0.75
*   **Normal:** 0.75 ≤ Ratio < 1.25
*   **Elevated:** 1.25 ≤ Ratio < 2.00
*   **Extreme:** Ratio ≥ 2.00

With a ratio of 0.97, the current volatility is near the historical median, placing the series firmly within the **normal** regime.

***

*Note: This analysis uses the historical BAA10Y change data provided up to the July 30, 2026 cutoff. The volatility ratio is defined as the current 30-day rolling standard deviation divided by the long-term median of those 30-day rolling standard deviations.*

## statistical_anomaly_detection

**Skill:** `statistical-analysis`  
**Question type:** `anomaly_detection`

### Response

To determine whether the latest 21-business-day BAA10Y spread-change observation is anomalous as of July 30, 2026, we examine the provided historical data.

### Analysis of BAA10Y Spread Change (21-Business-Day Horizon)

Based on the target series history, the most recent 21-business-day change observation (calculated as the difference between the spread on July 30, 2026, and the spread 21 business days prior) is **10.0 basis points**.

*   **Latest 21-business-day observation:** 10.0 bps
*   **Rolling standard deviation (based on recent history):** 28.51 bps
*   **Z-score:** 0.35
*   **Absolute z-score > 2.5:** No

### Conclusion

The latest 21-business-day BAA10Y spread-change observation is **not anomalous**. With an absolute z-score of 0.35, the current movement is well within the expected range of historical volatility.

***

*Note: Calculations are based on the available `target_history` provided in the data payload. As per the instructions, this analysis does not alter the supplied numerical data.*

## statistical_window_selection

**Skill:** `statistical-analysis`  
**Question type:** `analysis_window`

### Response

To determine the appropriate observation window for BAA10Y statistical analysis as of July 30, 2026, we examine the recent behavior of the BAA10Y daily spread changes.

### Observed Evidence
The BAA10Y daily spread change on July 30, 2026, was **10.0 bps**. Analyzing the most recent 45 business days of spread change data (the maximum window requested), the descriptive statistics are:

*   **Recent Median:** -3.0 bps
*   **Recent Standard Deviation (Volatility):** 5.75 bps

### Volatility Regime and Z-Score
The volatility regime is characterized as **Moderate**. 

To calculate the latest-observation z-score, we use the formula: $Z = (x - \mu) / \sigma$, where $x$ is the most recent observation (10.0), $\mu$ is the recent median (-3.0), and $\sigma$ is the recent standard deviation (5.75).

*   **Latest-Observation Z-Score:** (10.0 - (-3.0)) / 5.75 ≈ **2.26**

### Statistical Analysis Selection
*   **Selected Window:** **15 Observations**
*   **Recent Median (15-day):** 6.0 bps
*   **Recent Standard Deviation (15-day):** 2.27 bps
*   **Selection Reason:** The latest-observation z-score of 2.26 indicates that the most recent data point is significantly higher than the 45-day median, suggesting a meaningful shift in recent credit spread dynamics. In periods where the current observation deviates by more than two standard deviations from the longer-term trend, a shorter observation window (15 days) is prioritized to ensure the model reacts promptly to the recent increase in volatility and captures the evolving market regime.

### Limitations
This analysis is based on the provided historical series and reflects point-in-time statistics as of July 30, 2026. The volatility regime assessment is purely statistical and does not incorporate fundamental macroeconomic drivers, which may offer context for the observed spread widening.

## driver_movement_summary

**Skill:** `credit-driver-analysis`  
**Question type:** `driver_movements`

### Response

As of July 30, 2026, the following analysis summarizes BAA10Y credit and market drivers. 

### Statistical Window Selection
Based on a review of the volatility regimes within the target BAA10Y series (which has exhibited notable recent spikes in daily spread changes), a **30-observation window** is selected. This duration provides a sufficient balance between capturing recent market responsiveness and filtering out extreme transient noise, which the 15-day window might overemphasize and the 45-day window might overly smooth.

---

### BAA10Y Driver Summary (30-Observation Window)

| Driver | Treatment | Latest Value | Net Movement | Z-Score | Status |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **CPI (MoM LogDiff)** | Scaled | -0.0042 | -0.0089 | -1.12 | Scorable |
| **Dollar Index (Log Ret)** | Absolute | 0.0014 | 0.0014 | 0.65 | Scorable |
| **Fed Funds Rate** | Level | 3.63% | 0.00 | 0.15 | Scorable |
| **HYOAS (Observed)** | Spread Change | 3.00 bps | 4.00 bps | 0.88 | Scorable |
| **NASDAQ (Log Ret)** | Absolute | -0.0176 | -0.0194 | -1.24 | Scorable |
| **Oil (Log Ret)** | Absolute | -0.0145 | 0.0184 | 0.45 | Scorable |
| **Unemployment Rate** | Level | 4.20% | -0.10 | -1.45 | Context |
| **UST 10Y Level** | Level | 4.67% | 0.19 | 1.10 | Scorable |
| **UST 2Y10Y Spread** | Level | 0.45 | 0.14 | 1.35 | Scorable |
| **VIX Level** | Level | 20.66 | 4.07 | 1.62 | Scorable |

*Note: Movements reflect the change over the selected 30-observation window. Z-scores are calculated relative to the 30-day trailing distribution.*

---

### Economic Interpretation
*   **Credit Factors:** The increase in observed HYOAS and the spike in the VIX suggest a tightening of financial conditions and a modest increase in risk aversion over the last 30 days. This is generally consistent with widening corporate spreads.
*   **Macro Factors:** The cooling trend in CPI MoM and the decline in the unemployment rate provide mixed signals, reflecting a potential shift in the inflationary environment that may influence the Fed's stance despite the current stability in the Fed Funds rate.
*   **Market Dynamics:** The steepening of the 2Y10Y UST curve and the recent sell-off in the NASDAQ indicate a rotation in investor sentiment, potentially increasing the required risk premium for BAA-rated issuers.

### Conditional Scenarios
*   **Widening Scenario:** Credit spreads would likely see upward pressure if the VIX sustains levels above 20 and the NASDAQ continues its downward momentum, suggesting continued systemic risk aversion.
*   **Tightening Scenario:** Spreads may find support if the 10Y UST yield stabilizes and inflation data remains benign, potentially alleviating the upward pressure on corporate borrowing costs.

### Limitations
This analysis relies on recent realized data. It does not account for forward-looking central bank commentary or unpredictable exogenous shocks that may occur after July 30, 2026. The HYOAS proxy (HYG-DGS3) was not treated independently from observed HYOAS per analytical protocols.

## driver_evidence_translation

**Skill:** `credit-driver-analysis`  
**Question type:** `widening_tightening_evidence`

### Response

Based on the available market data as of July 30, 2026, the following analysis assesses the drivers for BAA10Y spread movements over a 21-business-day horizon.

### Statistical Analysis Window
To capture the most relevant recent market volatility and trend shifts, a **21-business-day (one-month) rolling window** is selected for evaluating the drivers of BAA10Y spread changes.

### Analysis of Market Drivers

Consistent with the requested analysis discipline, drivers are classified below. Where scorable, they have been assigned a qualitative credit score based on their observed recent momentum.

| Market Driver | Classification | Credit Score | Interpretation |
| :--- | :--- | :--- | :--- |
| **HYOAS (Observed)** | Widening | -2 | Sustained widening pressure over the recent window. |
| **VIX (Level)** | Widening | -2 | Elevated volatility level suggests increased risk aversion. |
| **UST 10Y Level** | Widening | -1 | Recent upward movement in yields correlates with higher discount rates and credit risk. |
| **Nasdaq (Returns)** | Widening | -1 | Equity weakness often precedes or coincides with credit spread widening. |
| **Dollar Index** | Neutral | 0 | Net change over the window shows no clear directional bias for credit spreads. |
| **CPI (MoM)** | Neutral | 0 | Recent inflation data has fluctuated without a definitive trend impact on credit spreads. |
| **Fed Funds Level** | Context-Only | N/A | Stability in the policy rate provides a backdrop for financing costs. |
| **Unemployment Rate**| Context-Only | N/A | Employment figures remain a foundational macro indicator. |
| **UST 2Y10Y Spread**| Context-Only | N/A | Steepening is monitored for cycle implications but currently serves as background. |
| **Oil (Returns)** | Context-Only | N/A | Commodity volatility is noted but does not currently override other credit-specific signals. |

*Note: In accordance with your instructions, HYOAS proxy signals and VIX returns were excluded to prevent double-counting of credit factor and volatility signals, respectively.*

### Economic Interpretation
The market environment as of late July 2026 is characterized by a "risk-off" posture. The combined signal of widening high-yield spreads and elevated equity volatility (VIX) is the primary driver of current BAA10Y dynamics. The rise in the UST 10Y yield, while often mixed in its implications, currently appears to be contributing to a more cautious credit environment.

### Conditional Scenarios
* **Widening becomes more likely if:** The observed widening in HYOAS continues to accelerate or if the VIX sustains its upward trajectory above current levels, signaling a broader market de-risking event.
* **Tightening becomes more likely if:** Equity markets demonstrate a sharp recovery, accompanied by a stabilization or reversal in the high-yield credit space, indicating a return of risk appetite.

### Limitations
This analysis is limited to the provided covariate history. It does not account for intraday data beyond the daily close on July 30, 2026, or non-modeled idiosyncratic events (e.g., sudden geopolitical shifts or unexpected corporate default announcements) that could impact BAA10Y spreads independent of these macroeconomic drivers.

## combined_driver_conclusion

**Skill:** `credit-driver-analysis`  
**Question type:** `combined_conclusion`

### Response

As of July 30, 2026, the analysis of BAA10Y spreads indicates a shift toward a **higher volatility regime**. Recent data shows elevated VIX levels (20.66) and notable spikes in observed HYOAS changes (9.56 bps on the proxy), suggesting that the market has transitioned from a stable environment to one characterized by increased sensitivity to macroeconomic and risk-factor inputs.

Given the recent instability, the appropriate analysis window is the **short-term (1–5 business days)** to capture the rapid transmission of market shocks, rather than relying on longer-term averages which would currently lag the heightened volatility. There are no clear indications of technical anomalies (e.g., liquidity vacuums) currently distorting the signal, though the regime change itself is the primary factor.

### Observed Evidence and Economic Interpretation
The market is currently transmitting a **widening signal** with a **moderate confidence level**.

*   **Strongest Widening Evidence:** The most significant indicator is the sharp rise in VIX (from 18.21 to 20.66) combined with the proxy HYOAS change of +9.56 bps on July 30. This confluence indicates a rapid retreat from risk across the credit spectrum, which typically flows into BAA corporate bond spreads.
*   **Strongest Tightening Evidence:** There is currently limited evidence for tightening. The primary offset is the recent stabilization of the Fed Funds rate (3.63%) and the slight cooling in month-over-month CPI inflation (as of July 15), which may temper upward pressure on Treasury yields.
*   **Important Context Factors:** 
    *   **Yield Curve:** The UST 2y10y spread jumped to 0.45, suggesting a potential steepening trend that bears watching, as it may signal changing expectations for long-term growth and monetary policy.
    *   **Equity Performance:** NASDAQ index performance remains inconsistent, providing a mixed sentiment signal that currently offers no clear support for credit tightening.

### Conditional Scenarios
*   **Widening would become more likely if:** The VIX continues to sustain levels above 20 and the HYOAS proxy records further consecutive daily increases, signaling sustained stress in the high-yield sector that spreads to investment-grade credit.
*   **Tightening would become more likely if:** Equity market volatility subsides, the UST 2y10y curve begins to normalize, and there is a sustained period of low-volatility, range-bound movement in the HYOAS proxy.

### Limitations
This analysis is constrained by the rapid evolution of the volatility regime. As of July 30, 2026, the reliance on high-frequency market proxies (HYG-DGS3) is essential, but these signals can be noisy. Longer-term macroeconomic fundamental data (such as quarterly earnings or comprehensive employment reports) are not updated with the same frequency and may not fully reflect the current intra-day risk-off sentiment.